In [ ]:
pip install --upgrade pip

In [ ]:
!pip install -q -U git+https://github.com/huggingface/transformers

In [ ]:
!pip install -q accelerate

In [ ]:
!pip install -q -i https://pypi.org/simple/ bitsandbytes

In [ ]:
pip install "pip<24.1"

In [ ]:
pip install -q pythainlp sacremoses

In [ ]:
!pip install fairseq==0.12.2 hydra-core==1.0.7 omegaconf

In [ ]:
!pip install peft bitsandbytes -q

In [ ]:
import torch
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

In [ ]:
torch.cuda.reset_max_memory_allocated()
torch.cuda.reset_max_memory_cached()

In [ ]:
import torch
from PIL import Image
from pathlib import Path
from tqdm import tqdm
from transformers import Blip2Processor, Blip2ForConditionalGeneration, BitsAndBytesConfig
from pythainlp.translate import Translate
import pandas as pd

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

processor = Blip2Processor.from_pretrained("Salesforce/blip2-opt-6.7b")
model = Blip2ForConditionalGeneration.from_pretrained(
   "Salesforce/blip2-opt-6.7b", torch_dtype=torch.float16
)
model.to(device)

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, 
    bnb_4bit_use_double_quant=False, 
    bnb_4bit_quant_type="nf4"
)

In [ ]:
translate_model = Translate('en', 'th', use_gpu=True)

In [ ]:
def process_image(path: Path, seen_ids: set):
    try:
        image = Image.open(path).convert("RGB")
        image_id = path.stem  # image_id example "00822"

        # check duplicate image_id will skip
        if image_id in seen_ids:
            print(f"Warning: Duplicate ID found, skipping {image_id}")
            return None
        seen_ids.add(image_id)

        # send image to module
        inputs = processor(image, return_tensors="pt").to(device, torch.float16)
        with torch.no_grad():  # release memory
            out = model.generate(**inputs, max_new_tokens=30, temperature=0.7)

        # translate english -> thai
        caption_en = processor.decode(out[0], skip_special_tokens=True)
        caption_th = translate_model.translate(caption_en)

        return (image_id, caption_th)

    except Exception as e:
        print(f"Error processing {path}: {e}")
        return None

def process_test_images(paths, batch_size=50):
    results = []
    seen_ids = set()  # check duplicate ID
    for i in tqdm(range(0, len(paths), batch_size), desc="Processing Test Images"):
        batch = paths[i : i + batch_size]
        batch_results = [process_image(p, seen_ids) for p in batch]
        batch_results = [r for r in batch_results if r]  # เอาเฉพาะข้อมูลที่ไม่ใช่ None
        results.extend(batch_results)
    return results

if __name__ == '__main__':
    test_path_coco = Path('/kaggle/input/super-ai-ss-5-img-captioning/test/test')

    # test images
    test_image_paths = list(test_path_coco.iterdir())

    print(f"📷 Found {len(test_image_paths)} test images")

    # process images
    results = process_test_images(test_image_paths, batch_size=50)

    # submission.csv
    submission = pd.DataFrame(results, columns=['image_id', 'caption'])
    submission.to_csv('submission.csv', index=False)

    print(f"Test processing complete! {len(submission)} test images saved to submission.csv")